In [1]:
!pip uninstall torchaudio -y

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128


In [2]:
!pip install datasets transformers torch pandas -q
!pip install --upgrade datasets torchvision accelerate transformers -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 124.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2

In [3]:
!pip install --upgrade torch torchvision transformers datasets -q

#Imports e Definição do Modelo

In [4]:
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer

# Definição do modelo

MODEL_NAME = "FacebookAI/roberta-base"

print(f"Carregando tokenizador para: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

Carregando tokenizador para: FacebookAI/roberta-base


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

#Carregamento e Amostragem

In [5]:
# Carrega o arquivo do Kaggle
dataset = load_dataset('csv', data_files="/content/IMDB Dataset.csv", split='train')

# Corte estratégico de dados para salvar tempo de processamento
# Selecionando 10.000 reviews aleatórias das 50.000 originais
dataset = dataset.shuffle(seed=42).select(range(10000))

print(f"Tamanho do dataset após corte: {len(dataset)} exemplos.")

Generating train split: 0 examples [00:00, ? examples/s]

Tamanho do dataset após corte: 10000 exemplos.


#Mapeamento e Tokenização

In [6]:
# 1. Converte 'positive' para 1 e 'negative' para 0
def map_labels(example):
    example['labels'] = 1 if example['sentiment'] == 'positive' else 0
    return example

dataset = dataset.map(map_labels)

# 2. Função de tokenização limitando a 128 tokens
def tokenize_function(examples):
    return tokenizer(
        examples["review"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

print("Iniciando tokenização em lote...")
tokenized_dataset = dataset.map(tokenize_function, batched=True)
print("Tokenização concluída!")

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Iniciando tokenização em lote...


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Tokenização concluída!


#Divisão e Preparação para o PyTorch

In [7]:
# Separa 80% para Treino e 20% para Teste/Validação
train_test_split = tokenized_dataset.train_test_split(test_size=0.2, seed=42)

# Divide os 20% restantes igualmente: Validação, Teste
test_val_split = train_test_split['test'].train_test_split(test_size=0.5, seed=42)

# Agrupa tudo em um único objeto DatasetDict
final_datasets = DatasetDict({
    'train': train_test_split['train'],
    'validation': test_val_split['train'],
    'test': test_val_split['test']
})

# Define o formato esperado pelos tensores do modelo
final_datasets.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

print("Dataset IMDB pronto para o Trainer!")
print(final_datasets)

Dataset IMDB pronto para o Trainer!
DatasetDict({
    train: Dataset({
        features: ['review', 'sentiment', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 8000
    })
    validation: Dataset({
        features: ['review', 'sentiment', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['review', 'sentiment', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 1000
    })
})


#Função de Métricas (Accuracy, Precision, Recall, F1)

In [8]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    # Calcula precision, recall e f1 usando a média binária (já que é positivo/negativo)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }

#Experimento 1

In [ ]:
import time
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

# ==========================================
# 1. RECARREGA O MODELO DO ZERO
# ==========================================
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# ==========================================
# 2. CONGELA AS CAMADAS (Exp 1)
# ==========================================
for param in model.base_model.parameters():
    param.requires_grad = False
print("Experimento 1: Apenas classificador liberado.")

# ==========================================
# 3. RECRIA O TRAINER E TREINA
# ==========================================

EPOCAS_ATUAIS = 2

training_args = TrainingArguments(
    output_dir="./resultados_exp1",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=EPOCAS_ATUAIS,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_datasets["train"],
    eval_dataset=final_datasets["validation"],
    compute_metrics=compute_metrics,
)

print(f"\nIniciando treinamento do Experimento 1 por {EPOCAS_ATUAIS} épocas...")
start_time = time.time()
trainer.train()
end_time = time.time()
print(f"Tempo total de treinamento (Exp 1): {(end_time - start_time) / 60:.2f} minutos")

# ==========================================
# 4. AVALIAÇÃO FINAL NO TESTE
# ==========================================
print("\n=== MÉTRICAS FINAIS EXP 1 (CONJUNTO DE TESTE) ===")
resultados_teste = trainer.evaluate(eval_dataset=final_datasets["test"])
for chave, valor in resultados_teste.items():
    if chave.startswith("eval_"):
        print(f"{chave.replace('eval_', '')}: {valor:.4f}")

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Experimento 1: Apenas classificador liberado.

Iniciando treinamento do Experimento 1 por 2 épocas...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.689288,0.682231,0.776000,0.826291,0.701195,0.758621
2,0.681657,0.678367,0.804000,0.772242,0.864542,0.815789


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Tempo total de treinamento (Exp 1): 2.15 minutos

=== MÉTRICAS FINAIS EXP 1 (CONJUNTO DE TESTE) ===


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.681657,0.680019,2,0.763000,0.740331,0.807229,0.772334


loss: 0.6800
accuracy: 0.7630
precision: 0.7403
recall: 0.8072
f1: 0.7723


#Experimento 2

In [10]:
import time
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

# ==========================================
# 1. RECARREGA O MODELO DO ZERO
# ==========================================
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# ==========================================
# 2. CONGELA AS CAMADAS (Exp 2 - Parcial)
# ==========================================
# Primeiro, congela tudo
for param in model.base_model.parameters():
    param.requires_grad = False

# Depois, libera apenas as duas últimas camadas
for name, param in model.named_parameters():
    if "layer.10" in name or "layer.11" in name:
        param.requires_grad = True
print("Experimento 2: Fine-tuning parcial liberado (Últimas camadas).")

# ==========================================
# 3. RECRIA O TRAINER E TREINA
# ==========================================
EPOCAS_ATUAIS = 2

training_args = TrainingArguments(
    output_dir="./resultados_exp2",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=EPOCAS_ATUAIS,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_datasets["train"],
    eval_dataset=final_datasets["validation"],
    compute_metrics=compute_metrics,
)

print(f"\nIniciando treinamento do Experimento 2 por {EPOCAS_ATUAIS} épocas...")
start_time = time.time()
trainer.train()
end_time = time.time()
print(f"Tempo total de treinamento (Exp 2): {(end_time - start_time) / 60:.2f} minutos")

# ==========================================
# 4. AVALIAÇÃO FINAL NO TESTE
# ==========================================
print("\n=== MÉTRICAS FINAIS EXP 2 (CONJUNTO DE TESTE) ===")
resultados_teste = trainer.evaluate(eval_dataset=final_datasets["test"])
for chave, valor in resultados_teste.items():
    if chave.startswith("eval_"):
        print(f"{chave.replace('eval_', '')}: {valor:.4f}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Experimento 2: Fine-tuning parcial liberado (Últimas camadas).

Iniciando treinamento do Experimento 2 por 2 épocas...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.413135,0.242697,0.904000,0.919421,0.886454,0.902637
2,0.277088,0.242771,0.903000,0.896282,0.912351,0.904245


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Tempo total de treinamento (Exp 2): 3.03 minutos

=== MÉTRICAS FINAIS EXP 2 (CONJUNTO DE TESTE) ===


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.277088,0.310430,2,0.873000,0.892178,0.847390,0.869207


loss: 0.3104
accuracy: 0.8730
precision: 0.8922
recall: 0.8474
f1: 0.8692


#Experimento 3

In [11]:
import time
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

# ==========================================
# 1. RECARREGA O MODELO DO ZERO
# ==========================================
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# ==========================================
# 2. LIBERA TODAS AS CAMADAS (Exp 3)
# ==========================================
for param in model.parameters():
    param.requires_grad = True
print("Experimento 3: Fine-tuning completo (Todas as camadas liberadas).")

# ==========================================
# 3. RECRIA O TRAINER E TREINA
# ==========================================
EPOCAS_ATUAIS = 2

training_args = TrainingArguments(
    output_dir="./resultados_exp3",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=EPOCAS_ATUAIS,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_datasets["train"],
    eval_dataset=final_datasets["validation"],
    compute_metrics=compute_metrics,
)

print(f"\nIniciando treinamento do Experimento 3 por {EPOCAS_ATUAIS} épocas...")
start_time = time.time()
trainer.train()
end_time = time.time()
print(f"Tempo total de treinamento (Exp 3): {(end_time - start_time) / 60:.2f} minutos")

# ==========================================
# 4. AVALIAÇÃO FINAL NO TESTE
# ==========================================
print("\n=== MÉTRICAS FINAIS EXP 3 (CONJUNTO DE TESTE) ===")
resultados_teste = trainer.evaluate(eval_dataset=final_datasets["test"])
for chave, valor in resultados_teste.items():
    if chave.startswith("eval_"):
        print(f"{chave.replace('eval_', '')}: {valor:.4f}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Experimento 3: Fine-tuning completo (Todas as camadas liberadas).

Iniciando treinamento do Experimento 3 por 2 épocas...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.367405,0.290089,0.895000,0.878095,0.918327,0.897760
2,0.219575,0.312764,0.898000,0.880228,0.922311,0.900778


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Tempo total de treinamento (Exp 3): 8.43 minutos

=== MÉTRICAS FINAIS EXP 3 (CONJUNTO DE TESTE) ===


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.219575,0.330785,2,0.876000,0.851504,0.909639,0.879612


loss: 0.3308
accuracy: 0.8760
precision: 0.8515
recall: 0.9096
f1: 0.8796
